In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Thư mục chứa dữ liệu của phần 4.3
DATA_DIR = Path("data_processed")

train_df = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=["date"]
)

test_df = pd.read_csv(
    DATA_DIR / "test.csv",
    parse_dates=["date"]
)

predictions = pd.read_csv(
    DATA_DIR / "rf_test_predictions.csv",
    parse_dates=["date"]
)

sigma_e = pd.read_csv(
    DATA_DIR / "sigma_e_by_item.csv"
)

# Ghép dự báo với doanh số thực tế để đánh giá mô phỏng
simulation_data = test_df[
    ["item_id", "date", "sales"]
].merge(
    predictions,
    on=["item_id", "date"],
    how="inner"
)

simulation_data = simulation_data.merge(
    sigma_e,
    on="item_id",
    how="left"
)

simulation_data = simulation_data.sort_values(
    ["item_id", "date"]
).reset_index(drop=True)

# Kiểm tra dữ liệu
if simulation_data.empty:
    raise ValueError("Không có dữ liệu chung giữa tập test và kết quả dự báo.")

if simulation_data["sigma_e"].isna().any():
    raise ValueError("Có sản phẩm chưa có sai số dự báo sigma_e.")

if simulation_data["y_pred"].isna().any():
    raise ValueError("Có giá trị dự báo bị thiếu.")

print("Số sản phẩm:", simulation_data["item_id"].nunique())
print("Ngày bắt đầu:", simulation_data["date"].min().date())
print("Ngày kết thúc:", simulation_data["date"].max().date())
print("Số dòng dữ liệu:", len(simulation_data))

display(simulation_data.head())

FileNotFoundError: [Errno 2] No such file or directory: 'data_processed\\train.csv'

In [ ]:
# Tham số mô phỏng
LEAD_TIME_DAYS = 6
REVIEW_INTERVAL_DAYS = 1
STOCK_COVER_DAYS = 3
Z = 1.65

# Tổng số ngày nhu cầu cần được đáp ứng
T = LEAD_TIME_DAYS + REVIEW_INTERVAL_DAYS

# Ngày cuối cùng có dữ liệu lịch sử
as_of_date = train_df["date"].max()

# Tính doanh số trung bình ngày của từng sản phẩm
history_start = as_of_date - pd.Timedelta(days=27)

history = train_df[
    train_df["date"].between(history_start, as_of_date)
]

average_sales = (
    history.groupby("item_id")["sales"]
    .mean()
    .rename("avg_daily_sales")
    .reset_index()
)

# Tồn kho ban đầu giả định bằng 3 ngày bán trung bình
initial_inventory = average_sales.copy()

initial_inventory["initial_stock"] = np.ceil(
    initial_inventory["avg_daily_sales"] * STOCK_COVER_DAYS
).astype(int)

# Kiểm tra các tham số
print("Thời gian giao hàng:", LEAD_TIME_DAYS, "ngày")
print("Chu kỳ xem xét:", REVIEW_INTERVAL_DAYS, "ngày")
print("Thời gian cần phủ:", T, "ngày")
print("Hệ số mức phục vụ z:", Z)

display(initial_inventory.head())

In [ ]:
def mo_phong_ton_kho(data, inventory, co_bo_sung=True):
    """
    Mô phỏng tồn kho theo ngày cho từng sản phẩm.

    data:
        Gồm nhu cầu thực tế, dự báo và sigma_e.
    inventory:
        Tồn kho ban đầu giả định của từng sản phẩm.
    co_bo_sung:
        True: có chính sách bổ sung hàng.
        False: không đặt thêm hàng.
    """

    data = data.copy()
    inventory = inventory.set_index("item_id")

    data = data.sort_values(
        ["item_id", "date"]
    ).reset_index(drop=True)

    ket_qua = []

    for item_id, nhom in data.groupby("item_id", sort=False):
        nhom = nhom.sort_values("date").reset_index(drop=True)

        if item_id not in inventory.index:
            continue

        ton_kho = int(
            inventory.loc[item_id, "initial_stock"]
        )

        sigma = float(nhom["sigma_e"].iloc[0])

        # Lưu các đơn hàng theo ngày dự kiến về kho
        don_hang_cho = {}

        for i in range(len(nhom)):
            ngay = nhom.loc[i, "date"]

            # 1. Nhận các đơn hàng đã đến ngày giao
            ton_kho += don_hang_cho.pop(i, 0)

            # 2. Tính vị thế tồn kho trước khi đặt đơn
            hang_dang_ve = sum(don_hang_cho.values())
            vi_the_ton_kho = ton_kho + hang_dang_ve

            luong_dat = 0
            nhu_cau_du_bao = 0
            ton_kho_an_toan = Z * sigma * np.sqrt(T)

            # Chỉ đặt hàng khi còn đủ T ngày dự báo
            # để tránh dùng khoảng dự báo không đầy đủ
            if co_bo_sung and i + T <= len(nhom):
                nhu_cau_du_bao = nhom.loc[
                    i:i + T - 1, "y_pred"
                ].sum()

                muc_ton_muc_tieu = (
                    nhu_cau_du_bao + ton_kho_an_toan
                )

                luong_dat = int(np.ceil(max(
                    0,
                    muc_ton_muc_tieu - vi_the_ton_kho
                )))

                # Đơn hàng về sau LEAD_TIME_DAYS ngày
                if luong_dat > 0:
                    ngay_nhan = i + LEAD_TIME_DAYS

                    if ngay_nhan < len(nhom):
                        don_hang_cho[ngay_nhan] = (
                            don_hang_cho.get(ngay_nhan, 0)
                            + luong_dat
                        )

            # 3. Đáp ứng nhu cầu thực tế trong ngày
            nhu_cau_thuc_te = max(
                0, int(nhom.loc[i, "sales"])
            )

            luong_ban = min(ton_kho, nhu_cau_thuc_te)
            thieu_hang = nhu_cau_thuc_te - luong_ban

            ton_kho -= luong_ban

            # 4. Lưu kết quả cuối ngày
            ket_qua.append({
                "item_id": item_id,
                "date": ngay,
                "demand": nhu_cau_thuc_te,
                "forecast": float(nhom.loc[i, "y_pred"]),
                "beginning_stock": ton_kho + luong_ban,
                "received_stock": (
                    don_hang_cho.get(i, 0)
                    if False else 0
                ),
                "sold": luong_ban,
                "lost_sales": thieu_hang,
                "ending_stock": ton_kho,
                "safety_stock": ton_kho_an_toan,
                "order_qty": luong_dat,
                "inventory_position": vi_the_ton_kho
            })

    return pd.DataFrame(ket_qua)


# Mô phỏng không bổ sung hàng
result_no_replenishment = mo_phong_ton_kho(
    simulation_data,
    initial_inventory,
    co_bo_sung=False
)

# Mô phỏng có bổ sung hàng theo dự báo
result_replenishment = mo_phong_ton_kho(
    simulation_data,
    initial_inventory,
    co_bo_sung=True
)

print("Đã hoàn thành mô phỏng tồn kho.")

In [ ]:
def tong_hop_hieu_qua(result, ten_phuong_an):
    tong_nhu_cau = result["demand"].sum()
    tong_ban = result["sold"].sum()
    tong_thieu = result["lost_sales"].sum()

    ty_le_dap_ung = (
        tong_ban / tong_nhu_cau * 100
        if tong_nhu_cau > 0 else 0
    )

    return {
        "Phương án": ten_phuong_an,
        "Tổng nhu cầu thực tế": tong_nhu_cau,
        "Lượng bán đáp ứng": tong_ban,
        "Lượng thiếu hàng": tong_thieu,
        "Tỷ lệ đáp ứng nhu cầu (%)": round(ty_le_dap_ung, 2),
        "Số dòng sản phẩm-ngày hết hàng": int(
            (
                (result["ending_stock"] == 0)
                & (result["demand"] > 0)
            ).sum()
        ),
        "Tồn kho cuối ngày trung bình": round(
            result["ending_stock"].mean(), 2
        ),
        "Tổng lượng đặt hàng": int(result["order_qty"].sum())
    }


summary = pd.DataFrame([
    tong_hop_hieu_qua(
        result_no_replenishment,
        "Không bổ sung hàng"
    ),
    tong_hop_hieu_qua(
        result_replenishment,
        "Có bổ sung hàng"
    )
])

display(summary)

In [ ]:
# Lưu kết quả chi tiết và bảng tổng hợp
result_no_replenishment.to_csv(
    DATA_DIR / "simulation_no_replenishment.csv",
    index=False,
    encoding="utf-8-sig"
)

result_replenishment.to_csv(
    DATA_DIR / "simulation_with_replenishment.csv",
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    DATA_DIR / "simulation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu kết quả mô phỏng tại:", DATA_DIR.resolve())
print("- simulation_no_replenishment.csv")
print("- simulation_with_replenishment.csv")
print("- simulation_summary.csv")

In [ ]:
import matplotlib.pyplot as plt

# So sánh lượng thiếu hàng giữa hai phương án
plt.figure(figsize=(8, 5))

plt.bar(
    summary["Phương án"],
    summary["Lượng thiếu hàng"]
)

plt.title("So sánh lượng thiếu hàng giữa hai phương án")
plt.ylabel("Số đơn vị hàng")
plt.xlabel("Phương án mô phỏng")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()


# So sánh tỷ lệ đáp ứng nhu cầu
plt.figure(figsize=(8, 5))

plt.bar(
    summary["Phương án"],
    summary["Tỷ lệ đáp ứng nhu cầu (%)"]
)

plt.title("So sánh tỷ lệ đáp ứng nhu cầu")
plt.ylabel("Tỷ lệ (%)")
plt.xlabel("Phương án mô phỏng")
plt.ylim(0, 100)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()